<a href="https://colab.research.google.com/github/huynhphatloi/semisub-cxr/blob/main/notebooks/colab_runner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Semi-Supervised Multi-Label CXR Classification — Colab Runner

Covers all 4 experimental settings:
1. **Supervised Baseline** (ImageNet pretrained)
2. **LVM-Med Fine-tuning** (medical-domain pretrained)
3. **Class-Wise Pseudo-Labeling** (teacher → student)
4. **Uncertainty-Filtered Pseudo-Labeling** (MC Dropout filtering)

All outputs are persisted to Google Drive so they survive session restarts.

## Cell 1 — Mount Google Drive

In [ ]:
from google.colab import drive
try:
    drive.mount('/content/drive')
except ValueError:
    print('Drive already mounted.')
print('Drive mounted at /content/drive')

## Cell 2 — Clone project from GitHub and install dependencies

Pulls the latest code directly from GitHub. Re-run this cell to get new changes.

In [ ]:
import os, sys, subprocess, shutil

PROJECT_DIR  = '/content/semisup-cxr'
GITHUB_REPO  = 'https://github.com/huynhphatloi/semisub-cxr.git'

# Always start from a known working directory
os.chdir('/content')

if os.path.isdir(PROJECT_DIR) and os.path.isdir(f'{PROJECT_DIR}/.git'):
    # Existing git repo — pull latest changes
    print('Pulling latest changes...')
    subprocess.run(['git', '-C', PROJECT_DIR, 'fetch', '--all'], check=True)
    subprocess.run(['git', '-C', PROJECT_DIR, 'reset', '--hard', 'origin/main'], check=True)
    print('Updated to latest commit.')
else:
    # Remove old non-git folder if it exists
    if os.path.isdir(PROJECT_DIR):
        print('Removing old project folder (not a git repo)...')
        shutil.rmtree(PROJECT_DIR)
    # Fresh clone
    print(f'Cloning {GITHUB_REPO} ...')
    subprocess.run(['git', 'clone', GITHUB_REPO, PROJECT_DIR], check=True)

# Add project to Python path
os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# Install dependencies
print('Installing dependencies...')
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    '-r', f'{PROJECT_DIR}/requirements.txt'
])
print('Done. Project ready at', PROJECT_DIR)

## Cell 2b — Download CheXpert dataset from Kaggle

Uses `kagglehub` to download and cache the dataset automatically.  
First time: you'll be prompted to log in to Kaggle.

In [ ]:
import os, subprocess, sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub'])
import kagglehub

DATASET_PATH = kagglehub.dataset_download('ashery/chexpert')
print(f'Dataset at: {DATASET_PATH}')
print(os.listdir(DATASET_PATH))

# CSV image paths start with CheXpert-v1.0-small/train/...
# Create symlink so they resolve correctly
small_link = f'{DATASET_PATH}/CheXpert-v1.0-small'
if not os.path.exists(small_link):
    os.symlink(DATASET_PATH, small_link)
    print('Created symlink for image path resolution')

## Cell 3 — Path and experiment configuration

**Edit the paths below to match your Drive layout.**

In [ ]:
import os, sys

DRIVE_ROOT     = '/content/drive/MyDrive'
PROJECT_DIR    = '/content/semisup-cxr'

# ── Dataset (CheXpert) ─────────────────────────────────────────────────────
# DATASET_PATH set by Cell 2b; fallback if Cell 2b was skipped
if 'DATASET_PATH' not in dir():
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub'])
    import kagglehub
    DATASET_PATH = kagglehub.dataset_download('ashery/chexpert')
print(f'Using dataset: {DATASET_PATH}')

# ── LVM-Med weights (optional, only needed for settings 2-4) ──────────────
LVMMED_WEIGHTS = f'{DRIVE_ROOT}/Projects/weights/lvmmed_resnet50.pth'

# ── All outputs go here (persisted to Drive) ───────────────────────────────
OUTPUT_DIR     = f'{DRIVE_ROOT}/Projects/semisub-cxr/outputs'

# ── Experiment parameters ──────────────────────────────────────────────────
LABELED_RATIO  = 0.05   # try 0.05, 0.10, 0.20
SEED           = 42

# ── Create output directories ──────────────────────────────────────────────
for d in [
    f'{OUTPUT_DIR}/splits',
    f'{OUTPUT_DIR}/checkpoints',
    f'{OUTPUT_DIR}/pseudo_labels',
    f'{OUTPUT_DIR}/results',
    f'{OUTPUT_DIR}/plots',
]:
    os.makedirs(d, exist_ok=True)

# ── Update YAML configs to point at your Drive paths ──────────────────────
import yaml

def patch_config(src_yaml, setting, output_dir):
    """Load a config template and patch paths for this Colab session."""
    with open(src_yaml) as f:
        cfg = yaml.safe_load(f)
    cfg['data']['dataset_path']   = DATASET_PATH
    cfg['split']['labeled_ratio'] = LABELED_RATIO
    cfg['split']['seed']          = SEED
    cfg['split']['splits_dir']    = f'{OUTPUT_DIR}/splits'
    cfg['output_dir']             = OUTPUT_DIR
    if setting in ('lvmmed', 'pseudo_label', 'uncertainty_filter'):
        cfg['model']['pretrained_weights_path'] = LVMMED_WEIGHTS
    patched = f'{PROJECT_DIR}/configs/_colab_{setting}.yaml'
    with open(patched, 'w') as f:
        yaml.dump(cfg, f)
    return patched

CONFIG_SUPERVISED   = patch_config(f'{PROJECT_DIR}/configs/supervised_baseline.yaml',  'supervised',          OUTPUT_DIR)
CONFIG_LVMMED       = patch_config(f'{PROJECT_DIR}/configs/lvmmed_finetune.yaml',       'lvmmed',              OUTPUT_DIR)
CONFIG_PSEUDO_LABEL = patch_config(f'{PROJECT_DIR}/configs/pseudo_label.yaml',          'pseudo_label',        OUTPUT_DIR)
CONFIG_UNCERTAINTY  = patch_config(f'{PROJECT_DIR}/configs/uncertainty_filter.yaml',    'uncertainty_filter',  OUTPUT_DIR)

def best_ckpt(setting):
    return f'{OUTPUT_DIR}/checkpoints/{setting}/{LABELED_RATIO}_{SEED}/best_checkpoint.pt'

print(f'Dataset     : {DATASET_PATH}')
print(f'Output dir  : {OUTPUT_DIR}')
print(f'Ratio={LABELED_RATIO}  Seed={SEED}')
print('Configs patched ✓')

## Cell 4 — Generate splits

In [ ]:
import subprocess, sys

result = subprocess.run([
    sys.executable, '-m', 'scripts.generate_splits',
    '--config', CONFIG_SUPERVISED
], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise RuntimeError(f'generate_splits failed (exit {result.returncode})')
print('Split generation complete.')

## Cell 5 — Train Supervised Baseline

In [ ]:
subprocess.run([
    sys.executable, '-m', 'scripts.train',
    '--config', CONFIG_SUPERVISED
], check=True)
print('Checkpoint:', best_ckpt('supervised'))

## Cell 6 — Train LVM-Med Fine-Tuned Model

> Skip this cell if you don't have LVM-Med weights — pseudo-labeling will use the supervised model as teacher instead.

In [ ]:
import os
if not os.path.isfile(LVMMED_WEIGHTS):
    print(f'[SKIP] LVM-Med weights not found at {LVMMED_WEIGHTS}')
    print('       Using supervised checkpoint as teacher instead.')
    TEACHER_CKPT = best_ckpt('supervised')
else:
    subprocess.run([
        sys.executable, '-m', 'scripts.train',
        '--config', CONFIG_LVMMED
    ], check=True)
    TEACHER_CKPT = best_ckpt('lvmmed')
    print('Checkpoint:', TEACHER_CKPT)

## Cell 7 — Generate pseudo-labels (confidence thresholding)

In [ ]:
subprocess.run([
    sys.executable, '-m', 'scripts.generate_pseudo_labels',
    '--config', CONFIG_PSEUDO_LABEL,
    '--teacher-checkpoint', TEACHER_CKPT
], check=True)
print('Pseudo-label artifacts saved.')

## Cell 8 — Train pseudo-label student

In [ ]:
subprocess.run([
    sys.executable, '-m', 'scripts.train',
    '--config', CONFIG_PSEUDO_LABEL
], check=True)
print('Checkpoint:', best_ckpt('pseudo_label'))

## Cell 9 — Generate uncertainty-filtered pseudo-labels (MC Dropout)

In [ ]:
subprocess.run([
    sys.executable, '-m', 'scripts.generate_pseudo_labels',
    '--config', CONFIG_UNCERTAINTY,
    '--teacher-checkpoint', TEACHER_CKPT
], check=True)
print('Uncertainty-filtered pseudo-labels saved.')

## Cell 10 — Train uncertainty-filtered student

In [ ]:
subprocess.run([
    sys.executable, '-m', 'scripts.train',
    '--config', CONFIG_UNCERTAINTY
], check=True)
print('Checkpoint:', best_ckpt('uncertainty_filter'))

## Cell 11 — Evaluate all checkpoints

In [ ]:
import os

settings_and_configs = [
    ('supervised',         CONFIG_SUPERVISED),
    ('lvmmed',             CONFIG_LVMMED),
    ('pseudo_label',       CONFIG_PSEUDO_LABEL),
    ('uncertainty_filter', CONFIG_UNCERTAINTY),
]

for setting, cfg_path in settings_and_configs:
    ckpt = best_ckpt(setting)
    if not os.path.isfile(ckpt):
        print(f'[SKIP] {setting}: no checkpoint at {ckpt}')
        continue
    print(f'Evaluating {setting}...')
    subprocess.run([
        sys.executable, '-m', 'scripts.evaluate',
        '--config', cfg_path,
        '--checkpoint', ckpt
    ], check=True)

print('All evaluations complete.')

## Cell 12 — Generate plots and tables

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display, Markdown
from src.evaluation.reporting import (
    generate_comparison_bar_chart,
    generate_per_class_auroc_plot,
    generate_results_table,
)

results_csv = f'{OUTPUT_DIR}/results/evaluation_results.csv'
plots_dir   = f'{OUTPUT_DIR}/plots'
os.makedirs(plots_dir, exist_ok=True)

label_names = [
    'Cardiomegaly', 'Pleural Effusion', 'Pneumothorax',
    'Consolidation', 'Atelectasis', 'Edema',
]

if not os.path.isfile(results_csv):
    print('No results CSV found. Run Cell 11 first.')
else:
    results_df = pd.read_csv(results_csv)

    # Generate files
    bar_path = generate_comparison_bar_chart(results_df, plots_dir)
    cls_path = generate_per_class_auroc_plot(results_df, plots_dir, label_names=label_names)
    md_path  = generate_results_table(results_df, plots_dir, fmt='markdown')
    tex_path = generate_results_table(results_df, plots_dir, fmt='latex')

    # Show results table inline
    display(Markdown('### Results Table'))
    display(Markdown(open(md_path).read()))

    # Show plots inline
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    axes[0].imshow(mpimg.imread(bar_path)); axes[0].axis('off'); axes[0].set_title('Macro AUROC Comparison')
    axes[1].imshow(mpimg.imread(cls_path)); axes[1].axis('off'); axes[1].set_title('Per-Class AUROC')
    plt.tight_layout()
    plt.show()

    print(f'\nAll outputs saved to {OUTPUT_DIR}/')
    print(f'  results/evaluation_results.csv')
    print(f'  plots/macro_auroc_comparison.png')
    print(f'  plots/per_class_auroc_comparison.png')
    print(f'  plots/results_table.md')
    print(f'  plots/results_table.tex')

## Cell 13 — Resume after session restart

If your Colab session disconnects, re-run Cells 1–3, then run only the cells you need to resume from.

In [ ]:
# Check what checkpoints already exist on Drive
import os

ckpt_root = f'{OUTPUT_DIR}/checkpoints'
print('Existing checkpoints:')
for setting in ['supervised', 'lvmmed', 'pseudo_label', 'uncertainty_filter']:
    ckpt = best_ckpt(setting)
    status = '✓ exists' if os.path.isfile(ckpt) else '✗ missing'
    print(f'  {setting:25s} {status}')

# To resume a specific training run, add to the config:
#   training:
#     resume_checkpoint: "/content/drive/MyDrive/semisup_outputs/checkpoints/supervised/0.05_42/last_checkpoint.pt"